# E-Commerce Dataset Cleaning

Clean, normalize, validate, and save the messy ecommerce orders dataset.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

input_path = "Ecommerce_Messy_25.csv"

df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (27, 9)


## Standardize Text and Missing Values

In [32]:
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in str_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
OrderID        0
OrderDate      0
CustomerID     0
Product        0
Quantity       0
UnitPrice      0
TotalAmount    0
PaymentMode    0
OrderStatus    0
dtype: int64


## Normalize Payment Mode

In [ ]:
payment_map = {
    "upi": "UPI",
    "wallet": "Wallet",
    "debit card": "Debit Card",
    "credit card": "Credit Card",
    "creditcard": "Credit Card",
    "cash on delivery": "Cash on Delivery",
}
df["PaymentMode"] = df["PaymentMode"].str.lower().map(payment_map)

print("Payment modes:", sorted(df["PaymentMode"].dropna().unique()))

Payment modes: ['Cash on Delivery', 'Credit Card', 'Debit Card', 'UPI', 'Wallet']


In [ ]:
df["OrderStatus"] = df["OrderStatus"].str.title()

print("Order statuses:", sorted(df["OrderStatus"].dropna().unique()))

## Normalize Order Status

In [34]:
df["OrderStatus"] = df["OrderStatus"].str.title()

print("Order statuses:", sorted(df["OrderStatus"].dropna().unique()))

Order statuses: ['Cancelled', 'Delivered', 'Returned']


## Normalize Unit Price

In [35]:
df["UnitPrice"] = (
    df["UnitPrice"]
    .astype("string")
    .str.replace(r"(?i)rs\.?\s*", "", regex=True)
    .str.replace("₹", "", regex=False)
    .str.replace("â‚¹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["UnitPrice"] = df["UnitPrice"].fillna(df["UnitPrice"].median())

print(df["UnitPrice"].describe())

count           27.0
mean     1750.851852
std      1185.284896
min            199.0
25%            799.0
50%           1799.0
75%           2499.0
max           3499.0
Name: UnitPrice, dtype: Float64


## Normalize Total Amount

In [36]:
df["TotalAmount"] = (
    df["TotalAmount"]
    .astype("string")
    .str.replace(r"(?i)rs\.?\s*", "", regex=True)
    .str.replace("₹", "", regex=False)
    .str.replace("â‚¹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["TotalAmount"] = pd.to_numeric(df["TotalAmount"], errors="coerce")

print(df["TotalAmount"].describe())

count           27.0
mean     3331.333333
std      2686.672434
min            199.0
25%           1099.0
50%           2499.0
75%           4998.0
max          10497.0
Name: TotalAmount, dtype: Float64


## Normalize Quantity

In [37]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
invalid_quantity_count = int((df["Quantity"] <= 0).sum())
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median()).astype("int64")

print("Invalid quantities corrected:", invalid_quantity_count)
print(df["Quantity"].describe())

Invalid quantities corrected: 2
count    27.000000
mean      1.962963
std       0.939782
min       1.000000
25%       1.000000
50%       2.000000
75%       2.500000
max       4.000000
Name: Quantity, dtype: float64


## Normalize Order Dates and Reconcile Totals

In [ ]:
df["OrderDate"] = pd.to_datetime(
    df["OrderDate"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)
print("Invalid or missing dates:", df["OrderDate"].isna().sum())
df["OrderDate"] = df["OrderDate"].dt.strftime("%Y-%m-%d")

Invalid or missing dates: 0
Totals reconciled: 5


In [ ]:
df["ExpectedTotal"] = (df["Quantity"] * df["UnitPrice"]).round(2)
total_mismatch_count = int((df["TotalAmount"] != df["ExpectedTotal"]).sum())
df["TotalAmount"] = df["ExpectedTotal"]
df = df.drop(columns="ExpectedTotal")
print("Totals reconciled:", total_mismatch_count)

## Remove Duplicate Orders and Validate

In [39]:
duplicate_count = int(df["OrderID"].duplicated().sum())
df = df.drop_duplicates(subset="OrderID", keep="first").reset_index(drop=True)

assert df["OrderID"].is_unique
assert df["OrderDate"].notna().all()
assert df["Quantity"].gt(0).all()
assert df["UnitPrice"].ge(0).all()
assert df["TotalAmount"].eq((df["Quantity"] * df["UnitPrice"]).round(2)).all()
assert df["PaymentMode"].isin(["UPI", "Wallet", "Debit Card", "Credit Card", "Cash on Delivery"]).all()
assert df["OrderStatus"].isin(["Delivered", "Returned", "Cancelled"]).all()

print("Duplicate orders removed:", duplicate_count)
print("Validated shape:", df.shape)
print("Missing values:")
print(df.isna().sum())

Duplicate orders removed: 2
Validated shape: (25, 9)
Missing values:
OrderID        0
OrderDate      0
CustomerID     0
Product        0
Quantity       0
UnitPrice      0
TotalAmount    0
PaymentMode    0
OrderStatus    0
dtype: int64


## Save the Cleaned Dataset

In [ ]:
output_path = "Ecommerce_Cleaned_25.csv"
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)
print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\E-Commerce\Ecommerce_Cleaned_25.csv
Saved shape: (25, 9)
